# ✈️ Airline Operations Intelligence
# Airport Reference Data Integration

## Purpose

This notebook starts **after flight-data cleaning**.

The cleaned flight dataset is imported as-is. We do not clean the flight data again here.

The objective is to:

1. Load the cleaned flight data.
2. Identify all airports used by the flights.
3. Load the OpenFlights airport reference data.
4. Check reference-data coverage.
5. Identify airports missing from the reference table.
6. Investigate the missing airports.
7. Add verified metadata for XWA and EAR.
8. Validate that every flight airport has a reference record.
9. Merge origin and destination airport metadata into the flight data.
10. Create the scheduled departure timestamp needed for weather integration.
11. Save the airport-enriched dataset.

### Important principle

**Do not delete flight records because a lookup/reference table is incomplete.**

Instead:

**Detect → Investigate → Supplement → Validate → Merge**

# PART 1 — IMPORT LIBRARIES

In [3]:
import pandas as pd
import numpy as np

In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# PART 2 — LOAD THE CLEANED FLIGHT DATA

Change the path below to your final cleaned flight CSV.

This notebook does not modify the raw dataset.

In [5]:
cleaned_path = r"D:\Data Analyst\EXCEL\api\processed\flights_2026_q1_cleaned.csv"

df = pd.read_csv(cleaned_path)

print("Cleaned flight data loaded successfully.")

Cleaned flight data loaded successfully.


In [6]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 1847242
Columns: 55


In [7]:
df.head()

,year,quarter,month,day_of_month,day_of_week,fl_date,mkt_unique_carrier,branded_code_share,mkt_carrier_airline_id,mkt_carrier,mkt_carrier_fl_num,origin,origin_city_name,origin_state_abr,origin_state_nm,dest,dest_city_name,dest_state_abr,dest_state_nm,crs_dep_time,dep_time,dep_delay,dep_delay_new,dep_del15,taxi_out,taxi_in,crs_arr_time,arr_time,arr_delay,arr_delay_new,arr_del15,cancelled,cancellation_code,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,flight_status,dep_time_at_cancellation,missing_operational_duration,extreme_dep_delay,extreme_arr_delay,is_cancelled,is_diverted,is_irregular_operation,is_departure_delayed,is_arrival_delayed,severe_departure_delay,severe_arrival_delay
0,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,1,JFK,"New York, NY",NY,New York,LAX,"Los Angeles, CA",CA,California,700,657.0,-3.0,0.0,0.0,67.0,11.0,1021,1104.0,43.0,43.0,1.0,0.0,NaN,0.0,381.0,427.0,349.0,2475.0,0.0,0.0,43.0,0.0,0.0,COMPLETED,NaN,False,False,False,0,0,0,0,1,0,0
1,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,10,LAX,"Los Angeles, CA",CA,California,JFK,"New York, NY",NY,New York,2130,2128.0,-2.0,0.0,0.0,16.0,6.0,551,520.0,-31.0,0.0,0.0,0.0,NaN,0.0,321.0,292.0,270.0,2475.0,NaN,NaN,NaN,NaN,NaN,COMPLETED,NaN,False,False,False,0,0,0,0,0,0,0
2,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,1002,MIA,"Miami, FL",FL,Florida,MSY,"New Orleans, LA",LA,Louisiana,2245,2252.0,7.0,7.0,0.0,16.0,3.0,2359,2356.0,-3.0,0.0,0.0,0.0,NaN,0.0,134.0,124.0,105.0,674.0,NaN,NaN,NaN,NaN,NaN,COMPLETED,NaN,False,False,False,0,0,0,0,0,0,0
3,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,1003,DEN,"Denver, CO",CO,Colorado,MIA,"Miami, FL",FL,Florida,2338,2338.0,0.0,0.0,0.0,13.0,6.0,527,508.0,-19.0,0.0,0.0,0.0,NaN,0.0,229.0,210.0,191.0,1709.0,NaN,NaN,NaN,NaN,NaN,COMPLETED,NaN,False,False,False,0,0,0,0,0,0,0
4,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,1004,BOS,"Boston, MA",MA,Massachusetts,CLT,"Charlotte, NC",NC,North Carolina,757,756.0,-1.0,0.0,0.0,26.0,4.0,1030,1023.0,-7.0,0.0,0.0,0.0,NaN,0.0,153.0,147.0,117.0,728.0,NaN,NaN,NaN,NaN,NaN,COMPLETED,NaN,False,False,False,0,0,0,0,0,0,0


# PART 3 — BASIC VALIDATION OF THE CLEANED INPUT

In [8]:
print("Origin missing values:", df["origin"].isna().sum())
print("Destination missing values:", df["dest"].isna().sum())

Origin missing values: 0
Destination missing values: 0


In [9]:
print("Unique origin airports:", df["origin"].nunique())
print("Unique destination airports:", df["dest"].nunique())

Unique origin airports: 362
Unique destination airports: 362


# PART 4 — FIND ALL AIRPORTS USED BY THE FLIGHT DATA

In [10]:
origin_airports = set(
    df["origin"].dropna().unique()
)

destination_airports = set(
    df["dest"].dropna().unique()
)

In [11]:
all_airports = origin_airports | destination_airports

print("Total unique airports used by flights:", len(all_airports))

Total unique airports used by flights: 362


In [12]:
print(sorted(all_airports))

['ABE', 'ABI', 'ABQ', 'ABR', 'ABY', 'ACT', 'ACV', 'ACY', 'ADK', 'ADQ', 'AEX', 'AGS', 'AKN', 'ALB', 'ALO', 'ALW', 'AMA', 'ANC', 'APN', 'ART', 'ASE', 'ATL', 'ATW', 'ATY', 'AUS', 'AVL', 'AVP', 'AZA', 'AZO', 'BDL', 'BET', 'BFF', 'BFL', 'BGM', 'BGR', 'BHM', 'BIH', 'BIL', 'BIS', 'BJI', 'BLI', 'BLV', 'BMI', 'BNA', 'BOI', 'BOS', 'BPT', 'BQK', 'BQN', 'BRD', 'BRO', 'BRW', 'BTM', 'BTR', 'BTV', 'BUF', 'BUR', 'BWI', 'BZN', 'CAE', 'CAK', 'CDC', 'CDV', 'CHA', 'CHO', 'CHS', 'CID', 'CIU', 'CKB', 'CLD', 'CLE', 'CLL', 'CLT', 'CMH', 'CMI', 'CMX', 'COD', 'COS', 'COU', 'CPR', 'CRP', 'CRW', 'CSG', 'CVG', 'CWA', 'CYS', 'DAB', 'DAL', 'DAY', 'DCA', 'DDC', 'DEC', 'DEN', 'DFW', 'DHN', 'DIK', 'DLG', 'DLH', 'DRO', 'DSM', 'DTW', 'DVL', 'EAR', 'EAT', 'EAU', 'ECP', 'EGE', 'EKO', 'ELM', 'ELP', 'ERI', 'ESC', 'EUG', 'EVV', 'EWN', 'EWR', 'EYW', 'FAI', 'FAR', 'FAT', 'FAY', 'FCA', 'FLG', 'FLL', 'FLO', 'FMN', 'FNT', 'FOD', 'FSD', 'FSM', 'FWA', 'GCC', 'GCK', 'GEG', 'GFK', 'GGG', 'GJT', 'GNV', 'GPT', 'GRB', 'GRI', 'GRK', 'GRR'

# PART 5 — LOAD THE OPENFLIGHTS AIRPORT REFERENCE

The OpenFlights `airports.dat` file does not contain a header row, so we provide the column names ourselves.

In [13]:
airport_path = r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\raw\airports\airports.dat"

airport_ref_all = pd.read_csv(
    airport_path,
    header=None
)

print("Airport reference loaded.")

Airport reference loaded.


In [14]:
print("Rows:", len(airport_ref_all))
print("Columns:", len(airport_ref_all.columns))

Rows: 7698
Columns: 14


In [15]:
airport_ref_all.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.081690,145.391998,5282,10,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,-5.207080,145.789001,20,10,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,-5.826790,144.296005,5388,10,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,-6.569803,146.725977,239,10,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,AYPY,-9.443380,147.220001,146,10,U,Pacific/Port_Moresby,airport,OurAirports


# PART 6 — ASSIGN AIRPORT REFERENCE COLUMN NAMES

In [16]:
airport_ref_all.columns = [
    "airport_id",
    "airport_name",
    "city",
    "country",
    "iata_code",
    "icao_code",
    "latitude",
    "longitude",
    "altitude",
    "timezone",
    "dst",
    "tz_database",
    "type",
    "source"
]

In [17]:
airport_ref_all.head()

,airport_id,airport_name,city,country,iata_code,icao_code,latitude,longitude,altitude,timezone,dst,tz_database,type,source
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.081690,145.391998,5282,10,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,-5.207080,145.789001,20,10,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,-5.826790,144.296005,5388,10,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,-6.569803,146.725977,239,10,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,AYPY,-9.443380,147.220001,146,10,U,Pacific/Port_Moresby,airport,OurAirports


# PART 7 — KEEP THE COLUMNS REQUIRED FOR THIS PROJECT

In [18]:
airport_ref = airport_ref_all[
    [
        "iata_code",
        "airport_name",
        "city",
        "country",
        "latitude",
        "longitude",
        "timezone"
    ]
].copy()

In [19]:
airport_ref.head()

,iata_code,airport_name,city,country,latitude,longitude,timezone
0,GKA,Goroka Airport,Goroka,Papua New Guinea,-6.081690,145.391998,10
1,MAG,Madang Airport,Madang,Papua New Guinea,-5.207080,145.789001,10
2,HGU,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,-5.826790,144.296005,10
3,LAE,Nadzab Airport,Nadzab,Papua New Guinea,-6.569803,146.725977,10
4,POM,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,-9.443380,147.220001,10


# PART 8 — STANDARDIZE AIRPORT CODES

Remove accidental spaces and make codes uppercase before matching.

In [20]:
df["origin"] = (
    df["origin"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["dest"] = (
    df["dest"]
    .astype("string")
    .str.strip()
    .str.upper()
)

In [21]:
airport_ref["iata_code"] = (
    airport_ref["iata_code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# PART 9 — CHECK FOR DUPLICATE AIRPORT CODES IN THE REFERENCE

In [22]:
duplicate_iata = airport_ref[
    airport_ref["iata_code"].duplicated(keep=False)
].sort_values("iata_code")

print("Duplicate IATA records:", len(duplicate_iata))

Duplicate IATA records: 1626


In [23]:
display(duplicate_iata.head(20))

,iata_code,airport_name,city,country,latitude,longitude,timezone
21,\N,Winnipeg / St. Andrews Airport,Winnipeg,Canada,50.056400,-97.032501,-6
6526,\N,Orillia Airport,Orillia,Canada,44.677656,-79.310217,-5
6525,\N,Edenvale Aerodrome,Edenvale,Canada,44.441101,-79.962799,-5
6523,\N,Kawartha Lakes (Lindsay) Airport,Lindsay,Canada,44.364700,-78.783897,-5
6522,\N,Stanhope Municipal Airport,Haliburton,Canada,45.110833,-78.640000,-5
6521,\N,Markham Airport,Markham,Canada,43.935799,-79.262199,-5
6520,\N,Huronia Airport,Midland,Canada,44.683300,-79.928299,-5
6509,\N,Warnervale Airport,Warnervale Airport,Australia,-33.240278,151.429722,10
6501,\N,Coral Creek Airport,Placida,United States,26.854500,-82.251198,-5
6499,\N,Fostoria Metropolitan Airport,Fostoria,United States,41.190800,-83.394501,-5


If duplicate IATA codes exist, investigate them before merging.

For this project, one IATA code should map to one airport reference record.

# PART 10 — FILTER REFERENCE DATA TO AIRPORTS USED BY THE FLIGHTS

In [24]:
airport_ref = airport_ref[
    airport_ref["iata_code"].isin(all_airports)
].copy()

In [25]:
print("Airport reference records used by this project:", len(airport_ref))

Airport reference records used by this project: 360


# PART 11 — CHECK FOR MISSING AIRPORT REFERENCE DATA

This is the important coverage check.

We compare airport codes in the flight data against airport codes in the reference table.

In [26]:
missing_airports = (
    all_airports
    - set(airport_ref["iata_code"].dropna())
)

print("Airports not found:", len(missing_airports))
print(sorted(missing_airports))

Airports not found: 2
['EAR', 'XWA']


# PART 12 — INVESTIGATE THE MISSING AIRPORTS

The original reference-data check identified:

- XWA
- EAR

We do not delete flights using these airports.

In [27]:
missing_airport_flights = df[
    df["origin"].isin(missing_airports)
    | df["dest"].isin(missing_airports)
]

In [28]:
print(
    "Flights affected by missing airport reference data:",
    len(missing_airport_flights)
)

Flights affected by missing airport reference data: 1447


In [29]:
display(
    missing_airport_flights[
        [
            "fl_date",
            "origin",
            "dest",
            "mkt_carrier",
            "mkt_carrier_fl_num"
        ]
    ].head(20)
)

,fl_date,origin,dest,mkt_carrier,mkt_carrier_fl_num
10045,2026-01-01,MSP,XWA,DL,4284
10046,2026-01-01,XWA,MSP,DL,4284
10048,2026-01-01,MSP,XWA,DL,4289
10054,2026-01-01,XWA,MSP,DL,4296
14883,2026-01-01,EAR,DEN,UA,5027
14900,2026-01-01,DEN,EAR,UA,5046
15004,2026-01-01,DEN,XWA,UA,5261
15037,2026-01-01,DEN,XWA,UA,5302
15096,2026-01-01,XWA,DEN,UA,5367
15178,2026-01-01,DEN,XWA,UA,5463


# PART 13 — COUNT MISSING AIRPORT USAGE

In [30]:
for airport in sorted(missing_airports):
    origin_count = (df["origin"] == airport).sum()
    destination_count = (df["dest"] == airport).sum()

    print(
        airport,
        "→ Origin flights:", origin_count,
        "| Destination flights:", destination_count
    )

EAR → Origin flights: 154 | Destination flights: 154
XWA → Origin flights: 570 | Destination flights: 569


# PART 14 — VERIFIED AIRPORT EXCEPTIONS

The initial OpenFlights reference does not contain XWA and EAR.

Verified metadata used to supplement the lookup table:

| IATA | Airport | City | Latitude | Longitude | IANA Timezone |
|---|---|---|---:|---:|---|
| XWA | Williston Basin International Airport | Williston | 48.2597836 | -103.7505567 | America/Chicago |
| EAR | Kearney Regional Airport | Kearney | 40.7270406 | -99.0067700 | America/Chicago |

These records are added because the airports exist in the flight data but were absent from the initial lookup source.

The values should be treated as reference-data supplementation, not flight-data cleaning.

In [31]:
missing_airport_records = pd.DataFrame({
    "iata_code": ["XWA", "EAR"],

    "airport_name": [
        "Williston Basin International Airport",
        "Kearney Regional Airport"
    ],

    "city": [
        "Williston",
        "Kearney"
    ],

    "country": [
        "United States",
        "United States"
    ],

    "latitude": [
        48.2597836,
        40.7270406
    ],

    "longitude": [
        -103.7505567,
        -99.0067700
    ],

    "timezone": [
        "America/Chicago",
        "America/Chicago"
    ]
})

In [32]:
missing_airport_records

,iata_code,airport_name,city,country,latitude,longitude,timezone
0,XWA,Williston Basin International Airport,Williston,United States,48.259784,-103.750557,America/Chicago
1,EAR,Kearney Regional Airport,Kearney,United States,40.727041,-99.006770,America/Chicago


# PART 15 — ADD THE VERIFIED AIRPORTS TO airport_ref

In [33]:
airport_ref = pd.concat(
    [
        airport_ref,
        missing_airport_records
    ],
    ignore_index=True
)

In [34]:
print(
    "Airport reference records after supplementation:",
    len(airport_ref)
)

Airport reference records after supplementation: 362


# PART 16 — VALIDATE XWA AND EAR

In [35]:
display(
    airport_ref[
        airport_ref["iata_code"].isin(["XWA", "EAR"])
    ]
)

,iata_code,airport_name,city,country,latitude,longitude,timezone
360,XWA,Williston Basin International Airport,Williston,United States,48.259784,-103.750557,America/Chicago
361,EAR,Kearney Regional Airport,Kearney,United States,40.727041,-99.006770,America/Chicago


In [36]:
xwa_ear_duplicates = airport_ref[
    airport_ref["iata_code"].isin(["XWA", "EAR"])
]["iata_code"].duplicated().sum()

print("XWA/EAR duplicate records:", xwa_ear_duplicates)

XWA/EAR duplicate records: 0


# PART 17 — RECHECK AIRPORT COVERAGE

This is the most important validation.

Expected result:

`Airports still missing: set()`

In [37]:
missing_airports = (
    all_airports
    - set(airport_ref["iata_code"].dropna())
)

print("Airports still missing:", len(missing_airports))
print(sorted(missing_airports))

Airports still missing: 0
[]


In [38]:
assert len(missing_airports) == 0

print("✓ Every flight airport exists in the airport reference.")

✓ Every flight airport exists in the airport reference.


# PART 18 — CHECK AIRPORT METADATA QUALITY

In [39]:
reference_columns = [
    "iata_code",
    "airport_name",
    "city",
    "latitude",
    "longitude",
    "timezone"
]

In [40]:
airport_ref[
    reference_columns
].isna().sum()

iata_code       0
airport_name    0
city            0
latitude        0
longitude       0
timezone        0
dtype: int64

## Check latitude and longitude ranges

In [41]:
invalid_latitude = (
    airport_ref["latitude"].notna()
    & (
        (airport_ref["latitude"] < -90)
        | (airport_ref["latitude"] > 90)
    )
)

invalid_longitude = (
    airport_ref["longitude"].notna()
    & (
        (airport_ref["longitude"] < -180)
        | (airport_ref["longitude"] > 180)
    )
)

In [42]:
print("Invalid latitude:", invalid_latitude.sum())
print("Invalid longitude:", invalid_longitude.sum())

Invalid latitude: 0
Invalid longitude: 0


# PART 19 — CREATE ORIGIN AIRPORT REFERENCE

In [43]:
origin_ref = airport_ref.copy()

In [44]:
origin_ref = origin_ref.rename(
    columns={
        "iata_code": "origin",
        "airport_name": "origin_airport_name",
        "city": "origin_city",
        "country": "origin_country",
        "latitude": "origin_latitude",
        "longitude": "origin_longitude",
        "timezone": "origin_timezone"
    }
)

In [45]:
origin_ref.head()

,origin,origin_airport_name,origin_city,origin_country,origin_latitude,origin_longitude,origin_timezone
0,PPG,Pago Pago International Airport,Pago Pago,American Samoa,-14.331000,-170.710007,-11
1,SPN,Saipan International Airport,Saipan,Northern Mariana Islands,15.119000,145.729004,10
2,GUM,Antonio B. Won Pat International Airport,Agana,Guam,13.483400,144.796005,10
3,STT,Cyril E. King Airport,St. Thomas,Virgin Islands,18.337299,-64.973396,-4
4,STX,Henry E Rohlsen Airport,St. Croix Island,Virgin Islands,17.701900,-64.798599,-4


# PART 20 — MERGE ORIGIN AIRPORT DATA

In [46]:
df_enriched = df.merge(
    origin_ref,
    on="origin",
    how="left",
    validate="many_to_one"
)

In [47]:
print("Rows after origin merge:", len(df_enriched))

Rows after origin merge: 1847242


`validate="many_to_one"` is used because many flights can belong to one airport reference record.

If the airport reference contains duplicate IATA codes, Pandas will raise an error instead of silently multiplying flight rows.

# PART 21 — CREATE DESTINATION AIRPORT REFERENCE

In [48]:
destination_ref = airport_ref.copy()

In [49]:
destination_ref = destination_ref.rename(
    columns={
        "iata_code": "dest",
        "airport_name": "dest_airport_name",
        "city": "dest_city",
        "country": "dest_country",
        "latitude": "dest_latitude",
        "longitude": "dest_longitude",
        "timezone": "dest_timezone"
    }
)

# PART 22 — MERGE DESTINATION AIRPORT DATA

In [50]:
df_enriched = df_enriched.merge(
    destination_ref,
    on="dest",
    how="left",
    validate="many_to_one"
)

In [51]:
print("Rows after destination merge:", len(df_enriched))

Rows after destination merge: 1847242


# PART 23 — VALIDATE ROW COUNT AFTER MERGES

Airport enrichment should add columns, not create or remove flight records.

In [52]:
assert len(df_enriched) == len(df)

print("✓ Flight row count preserved.")

✓ Flight row count preserved.


# PART 24 — VALIDATE ORIGIN METADATA

In [53]:
print(
    "Missing origin latitude:",
    df_enriched["origin_latitude"].isna().sum()
)

print(
    "Missing origin longitude:",
    df_enriched["origin_longitude"].isna().sum()
)

print(
    "Missing origin timezone:",
    df_enriched["origin_timezone"].isna().sum()
)

Missing origin latitude: 0
Missing origin longitude: 0
Missing origin timezone: 0


# PART 25 — VALIDATE DESTINATION METADATA

In [54]:
print(
    "Missing destination latitude:",
    df_enriched["dest_latitude"].isna().sum()
)

print(
    "Missing destination longitude:",
    df_enriched["dest_longitude"].isna().sum()
)

print(
    "Missing destination timezone:",
    df_enriched["dest_timezone"].isna().sum()
)

Missing destination latitude: 0
Missing destination longitude: 0
Missing destination timezone: 0


# PART 26 — CREATE SCHEDULED DEPARTURE TIME

The weather integration will need the scheduled departure timestamp.

The flight dataset contains:

- `fl_date`
- `crs_dep_time`

`crs_dep_time` uses HHMM format.

In [55]:
df_enriched["fl_date"] = pd.to_datetime(
    df_enriched["fl_date"],
    errors="coerce"
)

In [56]:
df_enriched["crs_dep_time"] = pd.to_numeric(
    df_enriched["crs_dep_time"],
    errors="coerce"
)

In [57]:
df_enriched["dep_hour"] = (
    df_enriched["crs_dep_time"] // 100
)

df_enriched["dep_minute"] = (
    df_enriched["crs_dep_time"] % 100
)

## Validate the departure-time components

In [58]:
invalid_departure_components = (
    df_enriched["dep_hour"].notna()
    & (
        (df_enriched["dep_hour"] < 0)
        | (df_enriched["dep_hour"] > 23)
        | (df_enriched["dep_minute"] < 0)
        | (df_enriched["dep_minute"] > 59)
    )
)

print(
    "Invalid scheduled departure components:",
    invalid_departure_components.sum()
)

Invalid scheduled departure components: 0


# PART 27 — BUILD SCHEDULED DEPARTURE TIMESTAMP

In [59]:
df_enriched["scheduled_departure"] = (
    df_enriched["fl_date"]
    + pd.to_timedelta(
        df_enriched["dep_hour"],
        unit="h"
    )
    + pd.to_timedelta(
        df_enriched["dep_minute"],
        unit="m"
    )
)

In [60]:
df_enriched[
    [
        "fl_date",
        "crs_dep_time",
        "scheduled_departure",
        "origin"
    ]
].head(20)

,fl_date,crs_dep_time,scheduled_departure,origin
0,2026-01-01,700,2026-01-01 07:00:00,JFK
1,2026-01-01,2130,2026-01-01 21:30:00,LAX
2,2026-01-01,2245,2026-01-01 22:45:00,MIA
3,2026-01-01,2338,2026-01-01 23:38:00,DEN
4,2026-01-01,757,2026-01-01 07:57:00,BOS
5,2026-01-01,2359,2026-01-01 23:59:00,SFO
6,2026-01-01,1926,2026-01-01 19:26:00,MIA
7,2026-01-01,1910,2026-01-01 19:10:00,DFW
8,2026-01-01,1646,2026-01-01 16:46:00,DFW
9,2026-01-01,2105,2026-01-01 21:05:00,DFW


# PART 28 — PREPARE THE WEATHER API INPUT

For the first weather integration, use the **origin airport** and scheduled departure.

The weather API input needs:

- Airport code
- Latitude
- Longitude
- Date
- Scheduled departure time

We will collect weather separately and later join using the nearest observation at or before scheduled departure.

In [61]:
weather_input = df_enriched[
    [
        "origin",
        "origin_latitude",
        "origin_longitude",
        "origin_timezone",
        "scheduled_departure"
    ]
].copy()

In [62]:
weather_input.head()

,origin,origin_latitude,origin_longitude,origin_timezone,scheduled_departure
0,JFK,40.639801,-73.778900,-5,2026-01-01 07:00:00
1,LAX,33.942501,-118.407997,-8,2026-01-01 21:30:00
2,MIA,25.793200,-80.290604,-5,2026-01-01 22:45:00
3,DEN,39.861698,-104.672997,-7,2026-01-01 23:38:00
4,BOS,42.364300,-71.005203,-5,2026-01-01 07:57:00


# PART 29 — REDUCE WEATHER API REQUESTS

Do not make one API request per flight.

Instead, identify unique airport/date combinations.

In [63]:
weather_input["weather_date"] = (
    weather_input["scheduled_departure"]
    .dt.date
)

In [64]:
weather_requests = (
    weather_input[
        [
            "origin",
            "origin_latitude",
            "origin_longitude",
            "origin_timezone",
            "weather_date"
        ]
    ]
    .drop_duplicates()
    .sort_values(["origin", "weather_date"])
    .reset_index(drop=True)
)

In [65]:
print(
    "Unique airport/date weather requests:",
    len(weather_requests)
)

Unique airport/date weather requests: 31837


In [66]:
weather_requests.head(20)

,origin,origin_latitude,origin_longitude,origin_timezone,weather_date
0,ABE,40.6521,-75.440804,-5,2026-01-01
1,ABE,40.6521,-75.440804,-5,2026-01-02
2,ABE,40.6521,-75.440804,-5,2026-01-03
3,ABE,40.6521,-75.440804,-5,2026-01-04
4,ABE,40.6521,-75.440804,-5,2026-01-05
5,ABE,40.6521,-75.440804,-5,2026-01-06
6,ABE,40.6521,-75.440804,-5,2026-01-07
7,ABE,40.6521,-75.440804,-5,2026-01-08
8,ABE,40.6521,-75.440804,-5,2026-01-09
9,ABE,40.6521,-75.440804,-5,2026-01-10


# PART 30 — SAVE AIRPORT-ENRICHED FLIGHTS

This is the output of the airport-reference phase.

It is ready for the next notebook where we collect and integrate historical weather.

In [67]:
df_enriched.shape

(1847242, 70)

In [68]:
output_path = r"D:\Data Analyst\EXCEL\api\processed\flights_2026_q1_airport.csv"

df_enriched.to_csv(
    output_path,
    index=False
)

print("Airport-enriched flight data saved:")
print(output_path)

Airport-enriched flight data saved:
D:\Data Analyst\EXCEL\api\processed\flights_2026_q1_airport.csv


In [76]:
combined_airports=df_enriched.head(100001)
combined_airports
# combined_airports.to_csv()

,year,quarter,month,day_of_month,day_of_week,fl_date,mkt_unique_carrier,branded_code_share,mkt_carrier_airline_id,mkt_carrier,mkt_carrier_fl_num,origin,origin_city_name,origin_state_abr,origin_state_nm,dest,dest_city_name,dest_state_abr,dest_state_nm,crs_dep_time,dep_time,dep_delay,dep_delay_new,dep_del15,taxi_out,taxi_in,crs_arr_time,arr_time,arr_delay,arr_delay_new,arr_del15,cancelled,cancellation_code,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,flight_status,dep_time_at_cancellation,missing_operational_duration,extreme_dep_delay,extreme_arr_delay,is_cancelled,is_diverted,is_irregular_operation,is_departure_delayed,is_arrival_delayed,severe_departure_delay,severe_arrival_delay,origin_airport_name,origin_city,origin_country,origin_latitude,origin_longitude,origin_timezone,dest_airport_name,dest_city,dest_country,dest_latitude,dest_longitude,dest_timezone,dep_hour,dep_minute,scheduled_departure
0,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,1,JFK,"New York, NY",NY,New York,LAX,"Los Angeles, CA",CA,California,700,657.0,-3.0,0.0,0.0,67.0,11.0,1021,1104.0,43.0,43.0,1.0,0.0,NaN,0.0,381.0,427.0,349.0,2475.0,0.0,0.0,43.0,0.0,0.0,COMPLETED,NaN,False,False,False,0,0,0,0,1,0,0,John F Kennedy International Airport,New York,United States,40.639801,-73.778900,-5,Los Angeles International Airport,Los Angeles,United States,33.942501,-118.407997,-8,7,0,2026-01-01 07:00:00
1,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,10,LAX,"Los Angeles, CA",CA,California,JFK,"New York, NY",NY,New York,2130,2128.0,-2.0,0.0,0.0,16.0,6.0,551,520.0,-31.0,0.0,0.0,0.0,NaN,0.0,321.0,292.0,270.0,2475.0,NaN,NaN,NaN,NaN,NaN,COMPLETED,NaN,False,False,False,0,0,0,0,0,0,0,Los Angeles International Airport,Los Angeles,United States,33.942501,-118.407997,-8,John F Kennedy International Airport,New York,United States,40.639801,-73.778900,-5,21,30,2026-01-01 21:30:00
2,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,1002,MIA,"Miami, FL",FL,Florida,MSY,"New Orleans, LA",LA,Louisiana,2245,2252.0,7.0,7.0,0.0,16.0,3.0,2359,2356.0,-3.0,0.0,0.0,0.0,NaN,0.0,134.0,124.0,105.0,674.0,NaN,NaN,NaN,NaN,NaN,COMPLETED,NaN,False,False,False,0,0,0,0,0,0,0,Miami International Airport,Miami,United States,25.793200,-80.290604,-5,Louis Armstrong New Orleans International Airport,New Orleans,United States,29.993401,-90.258003,-6,22,45,2026-01-01 22:45:00
3,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,1003,DEN,"Denver, CO",CO,Colorado,MIA,"Miami, FL",FL,Florida,2338,2338.0,0.0,0.0,0.0,13.0,6.0,527,508.0,-19.0,0.0,0.0,0.0,NaN,0.0,229.0,210.0,191.0,1709.0,NaN,NaN,NaN,NaN,NaN,COMPLETED,NaN,False,False,False,0,0,0,0,0,0,0,Denver International Airport,Denver,United States,39.861698,-104.672997,-7,Miami International Airport,Miami,United States,25.793200,-80.290604,-5,23,38,2026-01-01 23:38:00
4,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,1004,BOS,"Boston, MA",MA,Massachusetts,CLT,"Charlotte, NC",NC,North Carolina,757,756.0,-1.0,0.0,0.0,26.0,4.0,1030,1023.0,-7.0,0.0,0.0,0.0,NaN,0.0,153.0,147.0,117.0,728.0,NaN,NaN,NaN,NaN,NaN,COMPLETED,NaN,False,False,False,0,0,0,0,0,0,0,General Edward Lawrence Logan International Ai...,Boston,United States,42.364300,-71.005203,-5,Charlotte Douglas International Airport,Charlotte,United States,35.214001,-80.943100,-5,7,57,2026-01-01 07:57:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99996,2026,1,1,5,1,2026-01-05,UA,UA,19977,UA,641,IAD,"Washington, DC",VA,Virginia,BOS,"Boston, MA",MA,Massachusetts,900,1051.0,111.0,111.0,1.0,14.0,6.0,1037,1214.0,97.0,97.0,1.0,0.0,NaN,0.0,97.0,83.0,63.0,412.0,3.0,0.0,0.0,0.0,94.0,COMPLETED,NaN,False,False,False,0,0,0,1,1,1,1,Washington Dulles International Airport,Washington,United States,38.944500,-77.455803,-5,General Edward Lawrence Logan

In [77]:
combined_airports.to_csv(r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\processed\combined_airports.csv")

# PART 31 — SAVE THE AIRPORT REFERENCE TABLE

Save the supplemented reference table separately so the XWA/EAR additions are documented and reusable.

In [70]:
airport_output_path = r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\raw\airports\airport_reference.csv"

airport_ref.to_csv(
    airport_output_path,
    index=False
)

print("Airport reference saved:")
print(airport_output_path)

Airport reference saved:
D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\raw\airports\airport_reference.csv


In [71]:
airport_ref.head()

,iata_code,airport_name,city,country,latitude,longitude,timezone
0,PPG,Pago Pago International Airport,Pago Pago,American Samoa,-14.331000,-170.710007,-11
1,SPN,Saipan International Airport,Saipan,Northern Mariana Islands,15.119000,145.729004,10
2,GUM,Antonio B. Won Pat International Airport,Agana,Guam,13.483400,144.796005,10
3,STT,Cyril E. King Airport,St. Thomas,Virgin Islands,18.337299,-64.973396,-4
4,STX,Henry E Rohlsen Airport,St. Croix Island,Virgin Islands,17.701900,-64.798599,-4


# PART 32 — SAVE THE WEATHER REQUEST LIST

This file will be used by the next weather-API notebook.

In [72]:
weather_request_path =r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\raw\weather\weather_requests.csv"

weather_requests.to_csv(
    weather_request_path,
    index=False
)

print("Weather request list saved:")
print(weather_request_path)

Weather request list saved:
D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\raw\weather\weather_requests.csv


In [73]:
weather_requests.head()

,origin,origin_latitude,origin_longitude,origin_timezone,weather_date
0,ABE,40.6521,-75.440804,-5,2026-01-01
1,ABE,40.6521,-75.440804,-5,2026-01-02
2,ABE,40.6521,-75.440804,-5,2026-01-03
3,ABE,40.6521,-75.440804,-5,2026-01-04
4,ABE,40.6521,-75.440804,-5,2026-01-05


# FINAL VALIDATION

Before moving to the Weather API, confirm all important checks passed.

In [74]:
assert len(df_enriched) == len(df)

assert len(missing_airports) == 0

assert invalid_latitude.sum() == 0

assert invalid_longitude.sum() == 0

assert df_enriched["origin_latitude"].isna().sum() == 0

assert df_enriched["origin_longitude"].isna().sum() == 0

assert df_enriched["dest_latitude"].isna().sum() == 0

assert df_enriched["dest_longitude"].isna().sum() == 0

print("✓ All airport integration validations passed.")

✓ All airport integration validations passed.


# ✅ AIRPORT REFERENCE INTEGRATION COMPLETE

## What was done

1. Imported the cleaned flight dataset.
2. Identified all origin and destination airports.
3. Loaded the OpenFlights airport reference.
4. Standardized airport codes.
5. Checked duplicate airport mappings.
6. Checked reference-data coverage.
7. Identified XWA and EAR as missing reference records.
8. Investigated flights affected by the missing mappings.
9. Added verified metadata for XWA and EAR.
10. Revalidated airport coverage.
11. Merged origin airport metadata.
12. Merged destination airport metadata.
13. Confirmed the flight row count did not change.
14. Validated coordinates and metadata.
15. Created scheduled departure timestamps.
16. Created a unique airport/date weather-request table.
17. Saved the airport-enriched dataset.
18. Saved the supplemented airport reference.
19. Saved the weather API request list.

## Data-quality decision

> During reference-data validation, I identified airport codes present in the flight dataset but missing from the lookup source. Instead of dropping those flights, I investigated the coverage gap, supplemented the reference table with verified metadata, and revalidated the mapping before weather integration.

## Next phase

**Historical Weather API Integration**

The next notebook should:

**weather_requests.csv → Open-Meteo Historical API → cached weather data → weather table → nearest observation at or before scheduled departure → flights_enriched**